# Chapter 24 — Log Analysis with AI
## From Signal Noise to Root Cause — in Seconds

A 500-device network generates **5 to 10 million syslog messages per day.**
Of those, maybe 50 to 100 are significant. Maybe 5 require immediate action.
The rest is background noise.

Traditional approach: write rules. "BGP" + "Idle" = alert.
Problem: **the novel multi-hop routing loop you never wrote a rule for? Missed.**

AI changes the paradigm from rule-based filtering to **semantic pattern recognition**:

> Instead of "does this match a known bad pattern?"
> ask "does this make sense in context, and if not, what does the deviation suggest?"

Three categories of AI log analysis — each needs a different technique:

| Category | Task | Technique |
|----------|------|-----------|
| **1 — Classification** | What type of event? How urgent? | Zero-shot / few-shot LLM |
| **2 — Correlation** | Which events share a cause? | Temporal clustering + causal graph |
| **3 — Root Cause** | What originated the fault? | MapReduce + dependency traversal |

> **No extra libraries.** All demos use Python built-ins + the Anthropic API.

### Install
```
pip install anthropic
```


## Setup — API Client and Log Dataset

In [ ]:
import os, json, re, math, time
from anthropic import Anthropic
from collections import defaultdict

try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import getpass
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')
client = Anthropic()   # reads ANTHROPIC_API_KEY from env

HAIKU  = "claude-haiku-4-5-20251001"
SONNET = "claude-sonnet-4-20250514"

def ask(prompt, model=HAIKU, system="You are a network operations AI assistant."):
    r = client.messages.create(
        model=model, max_tokens=1024, system=system,
        messages=[{"role": "user", "content": prompt}],
    )
    return r.content[0].text.strip()

# ── Realistic mixed log dataset ───────────────────────────────────────────────
# 5-10M/day in production — this is a representative 60-second window
LOG_DATASET = [
    # ── Causal chain: link flap → OSPF → BGP → apps (14:32:15 – 14:32:28) ──
    "Feb 25 14:32:15 CORE-SW1 %LINEPROTO-5-UPDOWN: Line protocol on Interface Gi1/0/1, changed state to down",
    "Feb 25 14:32:15 CORE-SW1 %LINEPROTO-5-UPDOWN: Line protocol on Interface Gi1/0/1, changed state to up",
    "Feb 25 14:32:17 R1 %OSPF-5-ADJCHG: Process 1, Nbr 10.0.1.2 on Gi0/0 from FULL to INIT, interface down",
    "Feb 25 14:32:17 R2 %OSPF-5-ADJCHG: Process 1, Nbr 10.0.1.1 on Gi0/0 from FULL to INIT, interface down",
    "Feb 25 14:32:17 R3 %OSPF-5-ADJCHG: Process 1, Nbr 10.0.1.1 on Gi0/1 from FULL to INIT, interface down",
    "Feb 25 14:32:22 R3 %BGP-5-ADJCHANGE: neighbor 203.0.113.1 Down Peer closed the session",
    "Feb 25 14:32:22 R3 %BGP-3-NOTIFICATION: sent to neighbor 203.0.113.1 6/3 (Peer De-Configured)",
    "Feb 25 14:32:28 APP-MON1 APP_UNREACHABLE: Application CRM unreachable from probe node, latency >30s",
    "Feb 25 14:32:28 APP-MON1 APP_UNREACHABLE: Application ERP unreachable from probe node",
    # ── Security event cluster ────────────────────────────────────────────────
    "Feb 25 14:33:01 FW1 %SEC-6-IPACCESSLOGP: list SSH-GUARD denied tcp 198.51.100.22(4122) -> 10.0.0.5(22), 1 packets",
    "Feb 25 14:33:02 FW1 %SEC-6-IPACCESSLOGP: list SSH-GUARD denied tcp 198.51.100.22(4123) -> 10.0.0.5(22), 1 packets",
    "Feb 25 14:33:03 FW1 %SEC-6-IPACCESSLOGP: list SSH-GUARD denied tcp 198.51.100.22(4124) -> 10.0.0.5(22), 1 packets",
    # ── Normal background noise (should be filtered as low priority) ──────────
    "Feb 25 14:33:10 R1 %SYS-6-LOGGINGHOST_STARTSTOP: Logging to host 10.0.0.200 port 514 started",
    "Feb 25 14:33:11 NTP-SERVER %NTP-6-PEERREACH: Peer 192.0.2.99 reachable",
    "Feb 25 14:33:15 CORE-SW1 %SYS-5-CONFIG_I: Configured from console by netops on vty0",
    "Feb 25 14:33:20 R2 %OSPF-5-ADJCHG: Process 1, Nbr 10.0.1.1 on Gi0/0 from INIT to FULL, Loading Done",
    "Feb 25 14:33:21 R3 %OSPF-5-ADJCHG: Process 1, Nbr 10.0.1.1 on Gi0/1 from INIT to FULL, Loading Done",
    # ── Novel vendor-specific format (not standard Cisco IOS) ─────────────────
    "Feb 25 14:34:00 JUNIPER-PE1 rpd[1503]: BGP_STOP: EBGP peer 203.0.113.5 (AS 65000) stopped, hold timer expired",
    "Feb 25 14:34:01 ARISTA-LEAF2 Stp: %SPANTREE-6-INTERFACE_ADD: Interface Et1/1 added instance MST0",
]

print(f"Log dataset: {len(LOG_DATASET)} messages in a 2-minute window")
print(f"(In production this would be ~14,000+ messages for a 500-device network)")


---
## Demo 1 — Zero-Shot Log Classification

**Zero-shot classification**: give the LLM a taxonomy and ask it to classify
log messages with no examples. The model generalizes from training knowledge.

This solves the **vocabulary mismatch problem**:
- "Disk capacity critical" and "Storage space exhausted" are the same event
- A keyword rule misses one. An LLM understands both.

Same idea for network logs:
- `%OSPF-5-ADJCHG ... from FULL to INIT` and `BGP_STOP: hold timer expired`
  are both "protocol adjacency loss" — different syntax, same meaning.

The LLM also does **contextual severity** — not just event type, but "how bad
is this *right now* given what else is happening?"


In [ ]:
# ── Zero-Shot Log Classification ─────────────────────────────────────────────

# Our event taxonomy — what we care about
TAXONOMY = {
    "OSPF": "OSPF protocol events: adjacency changes, SPF runs, area transitions",
    "BGP":  "BGP protocol events: session state, route updates, notifications",
    "INTERFACE": "Physical/logical interface state changes: up/down/admin-down",
    "SECURITY": "Security events: ACL denies, authentication failures, scans",
    "APPLICATION": "Application or service availability events",
    "NORMAL": "Expected background events: NTP sync, config saves, logging, routine",
}

def classify_batch(logs: list[str]) -> list[dict]:
    """
    Zero-shot classification: classify all logs in one LLM call (efficient).
    Batch processing reduces API calls and latency.
    """
    log_block = "\n".join(f"{i+1}. {msg}" for i, msg in enumerate(logs))
    taxonomy_block = "\n".join(f"  {k}: {v}" for k, v in TAXONOMY.items())

    prompt = f"""Classify each of these network log messages.

Categories:
{taxonomy_block}

Priority levels:
  P1: Service outage or imminent risk, needs immediate action
  P2: Degradation or warning, needs attention within 1 hour
  P3: Informational, log and monitor

Logs to classify:
{log_block}

Return ONLY a JSON array, one object per log:
[
  {{
    "id": 1,
    "category": "OSPF",
    "priority": "P1",
    "reason": "OSPF adjacency lost — protocol reconvergence triggered"
  }},
  ...
]
"""
    raw = ask(prompt, model=SONNET,
              system="You are a network log classifier. Return only valid JSON.")
    match = re.search(r'\[.*\]', raw, re.DOTALL)
    if not match:
        return []
    try:
        return json.loads(match.group())
    except Exception:
        return []


classifications = classify_batch(LOG_DATASET)

# Display grouped by priority
by_priority = defaultdict(list)
for c in classifications:
    by_priority[c.get("priority", "?")].append(c)

for priority in ["P1", "P2", "P3"]:
    items = by_priority.get(priority, [])
    if not items:
        continue
    icon = {"P1": "🚨", "P2": "🔴", "P3": "ℹ"}.get(priority, "?")
    print(f"\n{icon} {priority} — {len(items)} event(s):")
    for item in items:
        log_idx = item.get("id", 1) - 1
        raw = LOG_DATASET[log_idx] if 0 <= log_idx < len(LOG_DATASET) else "?"
        print(f"  [{item.get('category','?')}] {raw[-60:]}...")
        print(f"    → {item.get('reason','')}")

p1_count = len(by_priority.get("P1", []))
p2_count = len(by_priority.get("P2", []))
p3_count = len(by_priority.get("P3", []))
print(f"\nSummary: {p1_count} P1 / {p2_count} P2 / {p3_count} P3 out of {len(LOG_DATASET)} logs")
print(f"Noise filtered: {p3_count}/{len(LOG_DATASET)} = {p3_count/len(LOG_DATASET)*100:.0f}% are background")


---
## Demo 2 — Few-Shot Classification for Novel/Vendor-Specific Events

**Zero-shot** works great for standard events (OSPF, BGP, Cisco IOS).
But what about **Juniper JunOS**, **Arista EOS**, or proprietary systems
that weren't well-represented in the model's training data?

**Few-shot classification**: give the LLM 3–5 labeled examples of the new format,
then ask it to classify new messages of the same type. No fine-tuning needed —
the model does **in-context learning**: it adapts its output based on examples
in the prompt itself.

```
Example 1: "rpd[1503]: BGP_STOP: peer 203.0.113.5 stopped" → BGP / P1
Example 2: "rpd[1503]: BGP_START: peer 203.0.113.5 established" → BGP / P3
Example 3: "mib2d: SNMP_TRAP_BGP_ESTABLISHED: peer up" → BGP / P3
                   ↓
Classify: "rpd[1602]: OSPF_NBR_DOWN: area 0.0.0.0 nbr 10.0.1.5 down" → ?
```


In [ ]:
# ── Few-Shot Classification for Novel Log Formats ────────────────────────────

# A few labeled examples teach the LLM the proprietary format
FEW_SHOT_EXAMPLES = [
    # Juniper JunOS rpd daemon logs
    {"log": "rpd[1503]: BGP_STOP: EBGP peer 203.0.113.5 (AS 65000) stopped, hold timer expired",
     "category": "BGP", "priority": "P1",
     "reason": "BGP session dropped — hold timer indicates connectivity or policy issue"},
    {"log": "rpd[1503]: BGP_START: EBGP peer 203.0.113.5 (AS 65000) established, 24842 received prefixes",
     "category": "BGP", "priority": "P3",
     "reason": "BGP session recovered — informational"},
    {"log": "rpd[1503]: OSPF_NBR_STATE_CHANGE: area 0.0.0.0, nbr 10.0.0.2 on ge-0/0/0, from Full to Init",
     "category": "OSPF", "priority": "P1",
     "reason": "OSPF adjacency lost — routing reconvergence"},
    # Arista EOS logs
    {"log": "Stp: %SPANTREE-6-INTERFACE_ADD: Interface Et1/2 added instance MST1",
     "category": "NORMAL", "priority": "P3",
     "reason": "STP topology change — expected during port addition"},
    {"log": "Stp: %SPANTREE-2-BPDUGUARD: BPDU guard triggered on Et1/5 — port disabled",
     "category": "SECURITY", "priority": "P2",
     "reason": "Potential loop or unauthorized switch connected — BPDU guard fired"},
]

# These are new messages in the same proprietary format to classify
NEW_MESSAGES = [
    "rpd[1602]: BGP_STOP: IBGP peer 10.0.0.9 (AS 65001) stopped, notification received (6/7)",
    "Stp: %SPANTREE-2-ROOTBRIDGE_CHANGE: Root bridge changed to Mac 00:50:56:aa:bb:cc on instance MST0",
    "rpd[1602]: OSPF_HELLO_DEAD: area 0.0.0.0, nbr 10.0.0.4 on ae0.100, hello packets missed",
    "mib2d[1204]: SNMP agent: authentication failure from 192.168.99.5, community string mismatch",
]

def few_shot_classify(examples: list[dict], new_msgs: list[str]) -> list[dict]:
    # Build the few-shot examples block
    examples_block = ""
    for i, ex in enumerate(examples, 1):
        examples_block += f"""
Example {i}:
  Log: {ex['log']}
  Category: {ex['category']} | Priority: {ex['priority']}
  Reason: {ex['reason']}
"""

    new_block = "\n".join(f"{i+1}. {msg}" for i, msg in enumerate(new_msgs))

    prompt = f"""Learn from these labeled examples, then classify the new messages.

{examples_block}

Now classify these NEW messages using the same format:
{new_block}

Return ONLY a JSON array:
[
  {{
    "id": 1,
    "log_snippet": "...",
    "category": "BGP",
    "priority": "P1",
    "reason": "..."
  }}
]
"""
    raw = ask(prompt, model=SONNET,
              system="You are a network log classifier. Learn from examples. Return only JSON.")
    match = re.search(r'\[.*\]', raw, re.DOTALL)
    if not match:
        return []
    try:
        return json.loads(match.group())
    except Exception:
        return []


results = few_shot_classify(FEW_SHOT_EXAMPLES, NEW_MESSAGES)

print("Few-shot classification results (Juniper + Arista logs):")
print("=" * 60)
print(f"Training examples: {len(FEW_SHOT_EXAMPLES)}")
print(f"New messages classified: {len(results)}")
print()
for i, (msg, result) in enumerate(zip(NEW_MESSAGES, results), 1):
    icon = {"P1": "🚨", "P2": "🔴", "P3": "ℹ"}.get(result.get("priority",""), "?")
    print(f"{i}. {msg[-70:]}")
    print(f"   {icon} [{result.get('category','?')}] {result.get('priority','?')} — {result.get('reason','')}")
    print()


---
## Demo 3 — TF-IDF Log Clustering: Group Similar Messages

Before an LLM can analyze 5 million log messages, we need to **group similar
messages together**. Clustering turns a flood of individual events into
a small number of meaningful groups.

**TF-IDF** (Term Frequency – Inverse Document Frequency) vectorizes each
log message into a numerical vector that captures which words are important
*for that specific message* compared to the rest of the corpus.

Then **cosine similarity** measures how close two vectors are in meaning.
Messages about the same type of event will cluster together — even if the
exact wording varies.

```
"OSPF adjacency from FULL to INIT" ──┐
"OSPF neighbor down on Gi0/0"    ──┤── OSPF cluster
"OSPF adj change: LOADING to FULL"──┘

"BGP session dropped hold timer" ──┐
"BGP STOP peer stopped"          ──┤── BGP cluster
"BGP notification sent"          ──┘
```

No GPU, no model — pure math. Then the LLM **names each cluster** in plain English.


In [ ]:
# ── TF-IDF Log Clustering (pure Python — no sklearn) ─────────────────────────

import math

def tokenize(text: str) -> list[str]:
    """Simple word tokenizer — lowercases and splits on non-alphanumeric chars."""
    return re.findall(r'[a-z][a-z0-9]+', text.lower())

def build_tfidf(documents: list[str]) -> tuple[list[dict], set]:
    """
    Compute TF-IDF vectors for all documents.
    Returns: list of {term: weight} dicts, vocabulary set.
    """
    N = len(documents)

    # Term frequency per document
    tf_docs = [Counter(tokenize(d)) for d in documents]
    # Count how many documents each term appears in
    df = Counter()
    for tf in tf_docs:
        df.update(tf.keys())

    # Compute TF-IDF
    tfidf = []
    vocab = set(df.keys())
    for tf in tf_docs:
        total = sum(tf.values()) or 1
        vec = {}
        for term, count in tf.items():
            tf_score  = count / total
            idf_score = math.log(N / (df[term] + 1)) + 1
            vec[term] = tf_score * idf_score
        tfidf.append(vec)

    return tfidf, vocab

def cosine_sim(a: dict, b: dict) -> float:
    """Cosine similarity between two TF-IDF vectors."""
    common = set(a) & set(b)
    dot    = sum(a[t] * b[t] for t in common)
    mag_a  = math.sqrt(sum(v*v for v in a.values()))
    mag_b  = math.sqrt(sum(v*v for v in b.values()))
    return dot / (mag_a * mag_b) if mag_a and mag_b else 0.0

def simple_cluster(documents: list[str], threshold: float = 0.25) -> dict[int, list[int]]:
    """
    Greedy clustering: assign each document to the first cluster
    whose centroid is similar enough. Create a new cluster otherwise.
    """
    tfidf, _ = build_tfidf(documents)
    clusters: dict[int, list[int]] = {}   # cluster_id -> [doc indices]
    centroids: list[dict] = []

    for i, vec in enumerate(tfidf):
        best_cluster = None
        best_sim     = threshold

        for cid, centroid in enumerate(centroids):
            sim = cosine_sim(vec, centroid)
            if sim > best_sim:
                best_sim     = sim
                best_cluster = cid

        if best_cluster is None:
            # Start a new cluster
            best_cluster = len(centroids)
            centroids.append(dict(vec))
            clusters[best_cluster] = []
        else:
            # Update centroid (running average)
            cid  = best_cluster
            n    = len(clusters[cid]) + 1
            for term in set(centroids[cid]) | set(vec):
                centroids[cid][term] = (
                    centroids[cid].get(term, 0) * (n-1) + vec.get(term, 0)
                ) / n

        clusters[best_cluster].append(i)

    return clusters

# Need Counter for tokenize()
from collections import Counter

# Strip timestamps to focus on the semantic content
clean_logs = [re.sub(r'^\S+\s+\d+:\d+:\d+\s+\S+\s+', '', log) for log in LOG_DATASET]

clusters = simple_cluster(clean_logs, threshold=0.15)

print(f"Clustered {len(LOG_DATASET)} logs into {len(clusters)} groups:")
print("=" * 60)

cluster_summaries = []
for cid, indices in sorted(clusters.items(), key=lambda x: -len(x[1])):
    msgs = [LOG_DATASET[i] for i in indices]
    print(f"\nCluster {cid} ({len(msgs)} message(s)):")
    for msg in msgs:
        print(f"  • {msg[-75:]}...")

    # Ask LLM to name this cluster
    name_prompt = f"""Name this cluster of network log messages in 5-8 words:
{chr(10).join(msgs[:3])}

Return ONLY the cluster name (e.g., "OSPF adjacency convergence events").
"""
    cluster_name = ask(name_prompt, model=HAIKU)
    print(f"  → Cluster name: {cluster_name}")
    cluster_summaries.append({"id": cid, "name": cluster_name, "count": len(msgs), "sample": msgs[0]})


---
## Demo 4 — Causal Correlation: Find the Root, Not Just the Symptoms

From the chapter: at 14:32:15 a core switch link flaps.
By 14:32:28, two applications are down. Thirteen seconds, five device types,
hundreds of messages — all correlated.

**Correlation ≠ causation.** The BGP drops and OSPF drops are correlated,
but neither caused the other. **Both were caused by the link flap.**

We need a **causal graph** — a map of what depends on what in the network:
```
CORE-SW1 Gi1/0/1 link
        │
        ↓ physical dependency
  OSPF adjacencies (R1, R2, R3)
        │
        ↓ routing dependency
  BGP sessions (R3 → ISP)
        │
        ↓ connectivity dependency
  Applications (CRM, ERP)
```

Traversing this graph backwards from observed symptoms → finds the root.


In [ ]:
# ── Causal Graph and Temporal Correlation ─────────────────────────────────────

# Network dependency graph: what depends on what
# Format: "symptom_device:event" -> ["root_cause_device:event", ...]
CAUSAL_GRAPH = {
    # OSPF adjacencies depend on physical links
    "R1:OSPF_DOWN":     ["CORE-SW1:LINK_DOWN"],
    "R2:OSPF_DOWN":     ["CORE-SW1:LINK_DOWN"],
    "R3:OSPF_DOWN":     ["CORE-SW1:LINK_DOWN"],
    # BGP depends on OSPF-learned routes to reach peer
    "R3:BGP_DOWN":      ["R3:OSPF_DOWN", "CORE-SW1:LINK_DOWN"],
    # Applications depend on BGP reachability
    "APP-MON1:APP_DOWN":["R3:BGP_DOWN"],
    # Security events are independent
    "FW1:ACL_DENY":     [],
}

# Map parsed events to causal graph keys
def event_to_causal_key(log: str) -> str | None:
    """Map a raw log message to a causal graph node identifier."""
    if "CORE-SW1" in log and "UPDOWN" in log and "down" in log.lower():
        return "CORE-SW1:LINK_DOWN"
    if "OSPF" in log and ("INIT" in log or "EXSTART" in log):
        device = log.split()[2]
        return f"{device}:OSPF_DOWN"
    if "BGP" in log and ("Down" in log or "STOP" in log):
        device = log.split()[2]
        return f"{device}:BGP_DOWN"
    if "APP_UNREACHABLE" in log:
        return "APP-MON1:APP_DOWN"
    if "IPACCESSLOGP" in log or "SSH-GUARD" in log:
        return "FW1:ACL_DENY"
    return None

def find_root_causes(observed_events: list[str]) -> dict:
    """
    Given observed symptoms, trace backwards through the causal graph
    to find the root cause(s) — nodes with no upstream dependencies.
    """
    # Identify which causal nodes are active
    active_nodes = set()
    for log in observed_events:
        key = event_to_causal_key(log)
        if key:
            active_nodes.add(key)

    # For each active node, trace back to find nodes with no parents
    # (nodes not caused by another active node)
    roots = set()
    for node in active_nodes:
        causes = CAUSAL_GRAPH.get(node, [])
        if not causes:
            roots.add(node)   # leaf in causal graph = root cause
        else:
            # Check if any cause is itself active (meaning it IS downstream)
            has_active_cause = any(c in active_nodes for c in causes)
            if not has_active_cause:
                roots.add(node)

    # Classify active nodes as root causes or downstream effects
    effects = active_nodes - roots

    return {
        "active_events": list(active_nodes),
        "root_causes": list(roots),
        "downstream_effects": list(effects),
    }

# Temporal clustering: group events by 5-second windows
def temporal_cluster(logs: list[str], window_seconds: int = 5) -> dict:
    """
    Group log events by time window. Events in the same window
    are candidates for a shared root cause.
    """
    TIME_PATTERN = re.compile(r'\d+:\d+:(\d+)')
    windows = defaultdict(list)

    for log in logs:
        m = TIME_PATTERN.search(log)
        if m:
            second = int(m.group(1))
            bucket = (second // window_seconds) * window_seconds
            windows[bucket].append(log)

    return dict(windows)


# Run analysis
print("CAUSAL CORRELATION ANALYSIS")
print("=" * 60)

# Step 1: Temporal clustering
time_windows = temporal_cluster(LOG_DATASET, window_seconds=5)
print(f"\nStep 1 — Temporal clustering (5-second windows):")
for window, msgs in sorted(time_windows.items()):
    print(f"  Window :xx:{window:02d} — {len(msgs)} events")

# Step 2: Root cause identification
print("\nStep 2 — Root cause identification:")
causal = find_root_causes(LOG_DATASET)
print(f"  Active event nodes: {causal['active_events']}")
print(f"  Root causes:        {causal['root_causes']}")
print(f"  Downstream effects: {causal['downstream_effects']}")

# Step 3: LLM explains the causal chain
chain_prompt = f"""Network incident analysis:

Root causes identified: {causal['root_causes']}
Downstream effects: {causal['downstream_effects']}

The causal dependency graph shows:
  Link flap → OSPF adjacency loss → BGP session drop → Application outage

Write a 4-sentence causal chain explanation:
1. What was the physical root cause?
2. How did it propagate through the protocol stack?
3. What services were impacted?
4. What is the recommended first fix?
"""
chain_explanation = ask(chain_prompt, model=SONNET)
print("\nStep 3 — Causal chain explanation:")
print(chain_explanation)


---
## Demo 5 — MapReduce Root Cause Analysis for Large Log Batches

The full log batch is too large to fit in one LLM prompt context.
**MapReduce** solves this:

```
5,000 log messages
        │
   ┌────┼────┐
  Chunk1  Chunk2  Chunk3   ← MAP: LLM summarizes each chunk independently
   │      │      │
   └──────┼──────┘
          │
     REDUCE: LLM combines all chunk summaries
          │
    Final Root Cause Report
```

Same pattern used for legal document summarization, code review, meeting notes.
For logs: each chunk summary preserves the key anomalies, the reduce step
finds the cross-chunk pattern that no single chunk would reveal.


In [ ]:
# ── MapReduce Root Cause Analysis ────────────────────────────────────────────

def map_chunk(chunk: list[str], chunk_id: int) -> str:
    """
    MAP step: summarize one chunk of log messages.
    Extract key events and any anomalies — discard routine noise.
    """
    log_block = "\n".join(chunk)
    prompt = f"""Analyze this chunk of network logs (chunk {chunk_id}).

Logs:
{log_block}

Summarize in 3-5 bullet points:
- What significant events occurred?
- Any anomalies or errors?
- Any causal relationships visible within this chunk?
- Approximate timeline of events?

Ignore routine informational messages (NTP sync, config saves, etc.)
Be specific: mention device names, protocols, and states.
"""
    return ask(prompt, model=HAIKU,
               system="You are a network log summarizer. Be concise and specific.")

def reduce_summaries(chunk_summaries: list[str], all_logs: list[str]) -> str:
    """
    REDUCE step: combine all chunk summaries into one root cause report.
    This is where cross-chunk correlation happens.
    """
    summaries_block = "\n\n".join(
        f"Chunk {i+1} summary:\n{s}" for i, s in enumerate(chunk_summaries)
    )
    prompt = f"""You are synthesizing summaries from {len(chunk_summaries)} log analysis chunks.

{summaries_block}

Write a final Root Cause Analysis report:

INCIDENT TIMELINE:
  (key events in chronological order)

ROOT CAUSE:
  (the single originating fault)

CAUSAL CHAIN:
  (step-by-step propagation)

IMPACTED SERVICES:
  (what stopped working and why)

RECOMMENDED ACTIONS:
  1. (immediate — stop the bleeding)
  2. (short-term — verify recovery)
  3. (long-term — prevent recurrence)

PRIORITY: P1/P2/P3
"""
    return ask(prompt, model=SONNET,
               system="You are a senior NOC engineer writing a root cause analysis report.")

def mapreduce_rca(logs: list[str], chunk_size: int = 6) -> str:
    """Full MapReduce pipeline for root cause analysis."""
    print(f"MapReduce RCA: {len(logs)} logs, chunk_size={chunk_size}")
    print("=" * 60)

    # Split into chunks
    chunks = [logs[i:i+chunk_size] for i in range(0, len(logs), chunk_size)]
    print(f"\n[MAP] Processing {len(chunks)} chunks in parallel...")

    # Map step
    summaries = []
    for i, chunk in enumerate(chunks, 1):
        print(f"  Chunk {i}/{len(chunks)}: {len(chunk)} logs → summarizing...")
        summary = map_chunk(chunk, i)
        summaries.append(summary)
        print(f"    ✓ done ({len(summary)} chars)")

    # Reduce step
    print(f"\n[REDUCE] Combining {len(summaries)} summaries into final report...")
    final_report = reduce_summaries(summaries, logs)

    print("\n" + "=" * 60)
    print("FINAL ROOT CAUSE ANALYSIS REPORT")
    print("=" * 60)
    print(final_report)
    return final_report


rca_report = mapreduce_rca(LOG_DATASET, chunk_size=5)


---
## Summary — AI Log Analysis Architecture

### The Three-Layer Stack

```
Layer 1: Clustering (TF-IDF)
  → Groups 5M logs/day into ~50 meaningful clusters
  → No LLM cost, pure math, runs at wire speed

Layer 2: Classification (Zero-shot / Few-shot LLM)
  → Labels each cluster: OSPF / BGP / Security / Normal
  → Assigns priority: P1 / P2 / P3
  → Solves vocabulary mismatch problem

Layer 3: Correlation + RCA (Causal graph + MapReduce LLM)
  → Temporal clustering finds co-occurring events
  → Causal graph separates root causes from downstream effects
  → MapReduce handles batches larger than context window
```

### Key Concepts

| Concept | What It Solves |
|---------|----------------|
| **Semantic embeddings** | Vocabulary mismatch — "disk capacity critical" = "storage exhausted" |
| **Zero-shot** | Known event types (OSPF, BGP) — no examples needed |
| **Few-shot** | Novel formats (Juniper, Arista) — 3-5 examples teach the LLM |
| **TF-IDF clustering** | Volume reduction — group before classifying |
| **Causal graph** | Separates root causes from downstream effects |
| **MapReduce** | Context window limit — chunk, map, reduce |

### Processing Economics

| Step | Volume | Tool | Cost |
|------|--------|------|------|
| Clustering | 5M logs/day | TF-IDF math | ~$0 |
| Cluster naming | ~50 clusters/day | Haiku | ~$0.001 |
| Classification | P1/P2 alerts only | Haiku | ~$0.01 |
| RCA synthesis | 1-5 incidents/day | Sonnet | ~$0.05 |

Total AI cost for a 500-device network: **under $0.10/day.**

### What's Next

- **Chapter 25**: Predictive Analytics — using historical log patterns and
  time-series data to predict failures *before* they happen, not just analyze them after

> The signal is there. Every link flap, every OSPF reconvergence, every BGP withdraw —
> it's all in the logs. AI doesn't give you more data. It gives you the ability
> to actually read it.
